# DreamDesk Stage 4: train the achiever inside the dream (GPU)

The policy practises reaching goals inside the world-model ensemble. No real app runs here; real tests happen on the laptop afterwards.

**How to run (foolproof):**
1. Add the dataset `dreamdesk-stage4` (the uploaded `dreamdesk_kaggle.zip`, unzipped by Kaggle) as input.
2. Settings: Accelerator **GPU T4 x1**, Internet off is fine.
3. Cells 1–3 were already checked once (GPU about 10 s per iteration). You can go straight to step 4.
4. Then **Save Version → Save & Run All (Commit)**. It runs in the background up to 12 h and keeps `/kaggle/working` as output.
5. If it stops before finishing: create a new version with the previous version's **output** added as an extra input. Cell 2 copies the saved runs back and every run resumes exactly where it stopped.
6. Download `achiever_results.zip` from the output.

In [ ]:
# 1. Configuration, as in experiments/stage4_prereg.md (Deviation 1: compute budget).
# Kaggle trains seed 2 of both arms; the laptop trains seeds 0 and 1.
ARMS = {"dream": 0.0, "penalty": 0.5}   # name -> disagreement penalty (0 = plain dream)
SEEDS = [2]
ITERATIONS = 1500                       # 150 PPO iterations per app
PARALLEL = 2                            # both runs at once on the GPU
THREADS = 2                             # CPU threads per training (Kaggle has 4 cores)

In [ ]:
# 2. Setup: find the input (following links, unzipping if needed), copy it to a writable place, restore earlier runs.
import glob, os, shutil, subprocess, sys, json, time, zipfile
def walk(top):
    for d, dirs, files in os.walk(top, followlinks=True):
        yield d, files
print("inputs:", {d: files[:6] for d, files in walk("/kaggle/input") if d.count("/") <= 5})
src = [d for d, files in walk("/kaggle/input") if "thresholds.json" in files]
if not src:  # Kaggle kept the zip as a file: unzip it ourselves
    zips = [os.path.join(d, f) for d, files in walk("/kaggle/input") for f in files if f.endswith(".zip")]
    assert zips, "no dataset attached: right panel -> Input -> Add Input -> Your Datasets -> dreamdesk-kaggle"
    zipfile.ZipFile(zips[0]).extractall("/kaggle/tmp_unzip")
    src = [d for d, files in walk("/kaggle/tmp_unzip") if "thresholds.json" in files]
assert src, "dataset found but thresholds.json missing inside it"
root = os.path.dirname(os.path.dirname(src[0]))          # the folder holding dreamdesk/, data/, runs/
W = "/kaggle/working/dd"
if not os.path.exists(W):
    shutil.copytree(root, W, ignore=shutil.ignore_patterns("ach"))
os.makedirs(f"{W}/runs/ach", exist_ok=True)
for d, files in walk("/kaggle/input"):                   # an earlier version's output: resume those runs
    if "state.pt" in files and "/runs/ach/" in d:
        dst = f"{W}/runs/ach/{os.path.basename(d)}"
        if not os.path.exists(f"{dst}/state.pt"):
            shutil.copytree(d, dst, dirs_exist_ok=True); print("restored", dst)
os.chdir(W)
# the uploaded code predates a fix: gi exists on Kaggle but without Atspi, which raises ValueError, not ImportError
p = "dreamdesk/atspi_backend.py"; s = open(p).read()
if "except ImportError:" in s:
    open(p, "w").write(s.replace("except ImportError:", "except (ImportError, ValueError):")); print("patched", p)
import torch
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")
assert torch.cuda.is_available(), "turn on the GPU accelerator in the notebook settings"
print(sorted(os.listdir("data/wm")), sorted(os.listdir("runs/wm")))


In [ ]:
# 3. Sanity check and speed: 4 iterations on GPU, compared with CPU, then the time estimate.
def bench(device):
    out = f"/tmp/bench_{device}"; shutil.rmtree(out, ignore_errors=True)
    t = time.time()
    r = subprocess.run([sys.executable, "-W", "ignore", "-m", "dreamdesk.achiever", "train", "--out", out, "--apps", "clocks", "files",
                        "--iterations", "4", "--threads", str(THREADS), "--device", device], capture_output=True, text=True)
    assert r.returncode == 0, r.stdout[-2000:] + r.stderr[-3000:]
    rows = [json.loads(l) for l in open(f"{out}/train_log.jsonl")]
    per_it = (rows[-1]["seconds"] - rows[0]["seconds"]) / (len(rows) - 1)
    print(f"{device}: {per_it:.1f} s per iteration (start-up {time.time() - t - rows[-1]['seconds']:.0f} s)")
    return per_it
gpu = bench("cuda"); cpu = bench("cpu")
runs = len(ARMS) * len(SEEDS)
print(f"plan: {runs} runs x {ITERATIONS} iterations; at {PARALLEL} in parallel about "
      f"{runs * ITERATIONS * gpu / PARALLEL / 3600:.1f} h on GPU (if parallel runs don't slow each other; check below)")

In [ ]:
# 4. Train every arm x seed. Finished runs are skipped, unfinished ones resume exactly.
from concurrent.futures import ThreadPoolExecutor
def train(job):
    arm, seed = job
    out = f"runs/ach/{arm}_seed{seed}"
    log = f"{out}/train_log.jsonl"
    if os.path.exists(f"{out}/done"):
        return f"{out}: already done"
    os.makedirs(out, exist_ok=True)
    with open(f"{out}/stdout.txt", "a") as f:
        r = subprocess.run([sys.executable, "-u", "-W", "ignore", "-m", "dreamdesk.achiever", "train", "--out", out,
                            "--seed", str(seed), "--penalty", str(ARMS[arm]), "--iterations", str(ITERATIONS),
                            "--threads", str(THREADS), "--device", "cuda", "--resume"], stdout=f, stderr=subprocess.STDOUT)
    if r.returncode == 0:
        open(f"{out}/done", "w").write("ok")
    return f"{out}: exit {r.returncode}"
jobs = [(a, s) for s in SEEDS for a in ARMS]
import threading
def progress():  # one line every 30 min, so a committed run's log shows it's alive
    while True:
        time.sleep(1800)
        for a, s in jobs:
            log = f"runs/ach/{a}_seed{s}/train_log.jsonl"
            n = sum(1 for _ in open(log)) if os.path.exists(log) else 0
            print(time.strftime("%H:%M"), f"{a}_seed{s}: {n}/{ITERATIONS} iterations", flush=True)
threading.Thread(target=progress, daemon=True).start()
with ThreadPoolExecutor(PARALLEL) as ex:
    for msg in ex.map(train, jobs):
        print(time.strftime("%H:%M"), msg, flush=True)

In [ ]:
# 5. Progress summary and the results archive to download.
for d in sorted(glob.glob("runs/ach/*")):
    rows = [json.loads(l) for l in open(f"{d}/train_log.jsonl")] if os.path.exists(f"{d}/train_log.jsonl") else []
    last = rows[-50:]
    print(f"{os.path.basename(d):16} {'done' if os.path.exists(d + '/done') else 'NOT DONE':8} it {rows[-1]['it'] + 1 if rows else 0:5}"
          f"  recent dream success {sum(r['dream_success'] for r in last) / max(1, len(last)):.3f}")
shutil.make_archive("/kaggle/working/achiever_results", "zip", W, "runs/ach")
print("saved /kaggle/working/achiever_results.zip")